# 05 — Out-of-domain evaluation (TweetEval)
Quiz the Amazon-trained baseline on tweets. Measure the generalization gap.
Input:  models/baseline/model.joblib  +  raw_data/tweeteval/sentiment_test.csv (raw 3-class)
Output: models/baseline/tweeteval_metrics.json

In [1]:
import pandas as pd, json, re, unicodedata
import joblib, numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from bparadigm.paths import BASELINE_MODEL, TWEETEVAL_TEST, BASELINE_METRICS, TWEETEVAL_METRICS

In [2]:
df = pd.read_csv(TWEETEVAL_TEST)
print(df.shape)
print(df["label"].value_counts().sort_index().to_dict())   # {0:neg, 1:neutral, 2:pos} — raw 3-class
df.head()

(12284, 2)
{0: 3972, 1: 5937, 2: 2375}


,text,label
0,@user @user what do these '1/2 naked pics' hav...,1
1,OH: “I had a blue penis while I was this” [pla...,1
2,"@user @user That's coming, but I think the vic...",1
3,I think I may be finally in with the in crowd ...,2
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",0


In [3]:
df = df[df["label"] != 1].copy()              # drop neutral (label 1) — no home in a binary model
df["label"] = df["label"].map({0: 0, 2: 1})   # neg 0 -> 0 (unchanged), pos 2 -> 1

In [4]:
print(df["label"].value_counts().sort_index().to_dict())   # expect {0: 3972, 1: 2375}
print("rows after dropping neutral:", len(df))

{0: 3972, 1: 2375}
rows after dropping neutral: 6347


In [5]:
_URL = re.compile(r"https?://\S+|www\.\S+"); _WS = re.compile(r"\s+")
def clean_text(t):                                  # IDENTICAL to the Amazon cleaning
    if not isinstance(t, str): return ""
    t = unicodedata.normalize("NFKC", t)
    t = _URL.sub(" ", t)
    return _WS.sub(" ", t).strip()

df["text"] = df["text"].map(clean_text)
df = df[df["text"].str.len() > 0]
print(df.shape)

(6347, 2)


In [6]:
pipe = joblib.load(BASELINE_MODEL)
y_true = df["label"]
y_pred = pipe.predict(df["text"])   # the model reads tweets through its Amazon vocabulary

In [7]:
majority = y_true.value_counts().idxmax()
floor_acc = accuracy_score(y_true, [majority] * len(y_true))
print(f"tweeteval floor: {floor_acc:.3f}")   # ~0.63, not 0.73

tweeteval floor: 0.626


In [8]:
acc    = accuracy_score(y_true, y_pred)
pos_f1 = f1_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
macro  = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"accuracy: {acc:.3f} | positive-F1: {pos_f1:.3f} | macro-F1: {macro:.3f}")
print(classification_report(y_true, y_pred, target_names=["Negative","Positive"], digits=3, zero_division=0))
print("confusion [true rows, pred cols] [Neg,Pos]:\n", confusion_matrix(y_true, y_pred, labels=[0,1]))

accuracy: 0.623 | positive-F1: 0.620 | macro-F1: 0.623
              precision    recall  f1-score   support

    Negative      0.824     0.506     0.627      3972
    Positive      0.498     0.819     0.620      2375

    accuracy                          0.623      6347
   macro avg      0.661     0.663     0.623      6347
weighted avg      0.702     0.623     0.624      6347

confusion [true rows, pred cols] [Neg,Pos]:
 [[2011 1961]
 [ 429 1946]]


In [9]:
indomain = json.loads((BASELINE_METRICS).read_text())["macro_f1"]
print(f"in-domain  (Amazon) macro-F1 : {indomain:.3f}")
print(f"out-domain (Tweet)  macro-F1 : {macro:.3f}")
print(f"generalization gap           : {indomain - macro:+.3f}")

out = {"dataset":"tweeteval_sentiment_test_binary", "n":int(len(df)),
       "accuracy":round(acc,4), "positive_f1":round(pos_f1,4), "macro_f1":round(macro,4),
       "floor_accuracy":round(floor_acc,4), "indomain_macro_f1":indomain,
       "generalization_gap":round(indomain-macro,4)}
(TWEETEVAL_METRICS).write_text(json.dumps(out, indent=2))
print("saved:", out)

in-domain  (Amazon) macro-F1 : 0.940
out-domain (Tweet)  macro-F1 : 0.623
generalization gap           : +0.317
saved: {'dataset': 'tweeteval_sentiment_test_binary', 'n': 6347, 'accuracy': 0.6234, 'positive_f1': 0.6195, 'macro_f1': 0.6234, 'floor_accuracy': 0.6258, 'indomain_macro_f1': 0.9403, 'generalization_gap': 0.3169}


Expectations:
- Expected macro-F1 BELOW your 0.940. The drop is the measurement, not a failure.
- A model near the ~0.63 floor here = it barely generalizes past guessing.
- Well above the floor but below 0.940 = it travels, but domain matters (as expected).
The fine-tuned RoBERTa must beat BOTH numbers and especially must shrink this gap, since that's where transformers earn their keep.